# Training runs

Trains all 11 configurations with seeds 0 to 4 (55 runs) under one budget: at most 10 epochs, early stopping with patience 3, best checkpoint by validation macro F1. Needs a GPU. One run took 20 to 40 minutes on a Kaggle Tesla T4.

We ran these cells in separate Kaggle sessions. A finished run is skipped and an interrupted one resumes from its last epoch, so the cells can be run in any order and repeated.

Inputs: the PlantVillage colour images and the split files (paths at the top of `run_experiments.py`, or set `PV_ROOT`, `SPLIT_ROOT` and `TRAIN_VAL_FILE`). Outputs go to `OUT_DIR` (default `/kaggle/working/revision`).

In [ ]:
import glob, os, shutil
SRC = "../../src"  # on Kaggle: attach this repository and point SRC to its src/ folder
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
for f in glob.glob(f"{SRC}/*.py"):
    shutil.copy(f, WORK)
%cd {WORK}

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

Quick check of the pipeline on 500 images and one epoch. The numbers mean nothing; it only shows that data loading, training and evaluation work.

In [ ]:
!python run_experiments.py --arch resnet50 --variant A0 --seeds 0 --smoke

## Architecture comparison

Frozen ImageNet backbone with a trainable head. For MobileNetV3-Large the whole classifier (960 to 1280 to 38) is trained.

In [ ]:
!python run_experiments.py --arch resnet50 --variant A0 --seeds 0 1 2 3 4
!python run_experiments.py --arch densenet121 --variant A0 --seeds 0 1 2 3 4
!python run_experiments.py --arch efficientnet_b0 --variant A0 --seeds 0 1 2 3 4
!python run_experiments.py --arch efficientnet_v2_s --variant A0 --seeds 0 1 2 3 4
!python run_experiments.py --arch mobilenet_v3_large --variant A0 --seeds 0 1 2 3 4

## Corruption-aware training ablation (ResNet50)

A1 adds Gaussian blur during training, A2 Gaussian noise, A3 brightness changes and A4 all three. Training corruptions are applied to the 256 x 256 image before resizing, in the same way as the test corruptions. A0 is the ResNet50 run above.

In [ ]:
!python run_experiments.py --arch resnet50 --variant A1 --seeds 0 1 2 3 4 --corrupt_before_resize
!python run_experiments.py --arch resnet50 --variant A2 --seeds 0 1 2 3 4 --corrupt_before_resize
!python run_experiments.py --arch resnet50 --variant A3 --seeds 0 1 2 3 4 --corrupt_before_resize
!python run_experiments.py --arch resnet50 --variant A4 --seeds 0 1 2 3 4 --corrupt_before_resize

## Class-balanced oversampling

Weighted sampling with replacement on top of A0 and A4. The number of distinct images drawn in each epoch is stored in the results file.

In [ ]:
!python run_experiments.py --arch resnet50 --variant A0 --seeds 0 1 2 3 4 --oversample
!python run_experiments.py --arch resnet50 --variant A4 --seeds 0 1 2 3 4 --oversample --corrupt_before_resize

In [ ]:
!ls revision/results | wc -l  # 55 when everything has finished